# Payment data quality
Synthetic data. Run all cells: inspect SQL, run cleaning, view Python chart. Expected: 12 input, 5 accepted, 6 review, 1 duplicate; $3,200 validated.

In [ ]:
from pathlib import Path
BASE = Path.cwd()
(BASE / 'messy_payments.csv').write_text('source_row,payment_id,department,payment_date,amount\n1,101, Health ,2026-09-01,1200\n2,101,health,2026-09-01,1200.00\n3,102,TRANSPORT,2026-09-02,850\n4,103,Health,2026-09-03,unknown\n5,104,Education,2026-09-04,500\n6,105,Health,2026-09-05,-75\n7,106,Transport,2026-02-30,300\n8,107,Education,2026-09-06,200\n9,107,Education,2026-09-06,250\n10,108,Health,2026-09-07,0\n11,,Health,2026-09-08,100\n12,109,Education,2026-09-08,650\n')


## SQL: inspect the cleaning rules

In [ ]:
sql = "-- SQLite SQL. Python registers strict date and currency validators.\nDROP TABLE IF EXISTS normalized;\nCREATE TABLE normalized AS\nSELECT *, NULLIF(TRIM(payment_id), '') AS business_id,\n       LOWER(TRIM(department)) AS clean_department,\n       valid_date(payment_date) AS clean_date,\n       cents(amount) AS amount_cents\nFROM raw_payments;\n\n-- Only identical normalized business records are deduplicated.\nDROP TABLE IF EXISTS ranked;\nCREATE TABLE ranked AS\nSELECT *, ROW_NUMBER() OVER (\n  PARTITION BY business_id, clean_department, payment_date, amount_cents,\n               CASE WHEN amount_cents IS NULL THEN amount ELSE '' END\n  ORDER BY CAST(source_row AS INTEGER)) AS duplicate_rank\nFROM normalized;\n\nDROP TABLE IF EXISTS duplicate_log;\nCREATE TABLE duplicate_log AS SELECT * FROM ranked WHERE duplicate_rank > 1;\n\n-- Different versions of one payment ID require review, not guessing.\nDROP TABLE IF EXISTS conflicts;\nCREATE TABLE conflicts AS\nSELECT business_id FROM ranked\nWHERE duplicate_rank = 1 AND business_id IS NOT NULL\nGROUP BY business_id HAVING COUNT(*) > 1;\n\nDROP TABLE IF EXISTS classified;\nCREATE TABLE classified AS\nSELECT *, CASE\n WHEN business_id IS NULL THEN 'Missing payment ID'\n WHEN business_id IN (SELECT business_id FROM conflicts) THEN 'Conflicting payment ID'\n WHEN clean_department NOT IN ('health','transport','education') THEN 'Unknown department'\n WHEN clean_date IS NULL THEN 'Invalid date'\n WHEN amount_cents IS NULL THEN 'Invalid amount'\n WHEN amount_cents < 0 THEN 'Negative amount: review'\n ELSE 'Accepted' END AS disposition\nFROM ranked WHERE duplicate_rank = 1;\n\nDROP TABLE IF EXISTS rejected_payments;\nCREATE TABLE rejected_payments AS SELECT * FROM classified WHERE disposition <> 'Accepted';\nDROP TABLE IF EXISTS clean_payments;\nCREATE TABLE clean_payments AS\nSELECT source_row, business_id AS payment_id, clean_department AS department,\n       clean_date AS payment_date, amount_cents\nFROM classified WHERE disposition = 'Accepted';\nDROP TABLE IF EXISTS payment_summary;\nCREATE TABLE payment_summary AS\nSELECT department, COUNT(*) AS payment_count, SUM(amount_cents) AS total_cents\nFROM clean_payments GROUP BY department;\n"
print(sql)
(Path.cwd() / 'clean_payments.sql').write_text(sql)


## Python: run the pipeline and display the chart

In [ ]:
from pathlib import Path
import csv, sqlite3
from datetime import date
from decimal import Decimal, InvalidOperation
import matplotlib.pyplot as plt

BASE = Path.cwd()
OUT = BASE / 'outputs'
OUT.mkdir(exist_ok=True)

def cents(value):
    # Exact cents; never silently round or use floating-point money.
    try:
        number = Decimal(str(value).strip())
        if not number.is_finite() or abs(number) > Decimal('1000000000000'):
            return None
        scaled = number * 100
        return int(scaled) if scaled == scaled.to_integral_value() else None
    except (InvalidOperation, ValueError, OverflowError):
        return None

def valid_date(value):
    try:
        text = str(value).strip()
        result = date.fromisoformat(text)
        return text if result.isoformat() == text else None
    except ValueError:
        return None

con = sqlite3.connect(':memory:')
con.create_function('cents', 1, cents)
con.create_function('valid_date', 1, valid_date)
con.execute('CREATE TABLE raw_payments(source_row TEXT, payment_id TEXT, department TEXT, payment_date TEXT, amount TEXT)')
with (BASE / 'messy_payments.csv').open(newline='') as file:
    records = list(csv.DictReader(file))
con.executemany('INSERT INTO raw_payments VALUES (?,?,?,?,?)', [tuple(r.values()) for r in records])
con.executescript((BASE / 'clean_payments.sql').read_text())

def count(table):
    return con.execute('SELECT COUNT(*) FROM ' + table).fetchone()[0]

raw_count, duplicates, rejected, accepted = [count(t) for t in
    ['raw_payments', 'duplicate_log', 'rejected_payments', 'clean_payments']]
total = con.execute('SELECT SUM(amount_cents) FROM clean_payments').fetchone()[0]
assert raw_count == duplicates + rejected + accepted
assert (raw_count, duplicates, rejected, accepted, total) == (12, 1, 6, 5, 320000)
assert cents('1.005') is None and cents('NaN') is None
assert valid_date('2026-02-30') is None
assert con.execute('SELECT COUNT(DISTINCT payment_id) FROM clean_payments').fetchone()[0] == accepted
# Re-running the transformations must reproduce the same result.
before = con.execute('SELECT * FROM clean_payments ORDER BY payment_id').fetchall()
con.executescript((BASE / 'clean_payments.sql').read_text())
assert con.execute('SELECT * FROM clean_payments ORDER BY payment_id').fetchall() == before

for table in ['clean_payments','rejected_payments','duplicate_log','payment_summary']:
    cursor = con.execute('SELECT * FROM ' + table)
    with (OUT / (table + '.csv')).open('w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([d[0] for d in cursor.description])
        writer.writerows(cursor)

summary = con.execute('SELECT department,total_cents FROM payment_summary ORDER BY total_cents DESC').fetchall()
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), layout='constrained')
fig.patch.set_facecolor('#f7f9fc')
axes[0].bar(['Accepted','Needs review','Duplicate'], [accepted,rejected,duplicates], color=['#347f83','#bd8850','#8d94a5'])
axes[0].set(title='Every source row is accounted for', ylabel='Source records', ylim=(0,8))
for i,v in enumerate([accepted,rejected,duplicates]): axes[0].text(i,v+.15,str(v),ha='center')
axes[1].barh([r[0].title() for r in summary], [r[1]/100 for r in summary], color='#547da6')
axes[1].set(title='Validated payments by department', xlabel='US dollars', xlim=(0,1550))
axes[1].invert_yaxis()
for i,(_,v) in enumerate(summary): axes[1].text(v/100+20,i,format(v/100,',.0f'),va='center')
fig.suptitle('Payment data quality | Synthetic demonstration',fontsize=15,fontweight='bold')
fig.savefig(OUT / 'payment_quality.png',dpi=180)
plt.show()
print(f'{raw_count} input = {accepted} accepted + {rejected} review + {duplicates} duplicate')
print(f'Validated total: ${total/100:,.2f}. All checks passed.')
